con reporte procesado

In [1]:
!pip install pymongo pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\34643\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
!pip install lxml

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\34643\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pymongo import MongoClient

# Conectar a MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["sistema_medico_procesado"]
coleccion = db["reportes_raw1"]

# Leer el CSV por bloques (chunks) para no saturar la memoria RAM
chunk_size = 10000 
for chunk in pd.read_csv("E://osley//Barecelona activa//DICONLINE//BD DICONLINE COPIA//reportes_procesados.csv", chunksize=chunk_size):
    # Convertir el bloque de DataFrame a diccionario de Python
    registros = chunk.to_dict(orient="records")
    # Insertar masivamente en MongoDB
    coleccion.insert_many(registros)

print("¡Medio millón de registros importados con éxito!")


¡Medio millón de registros importados con éxito!


In [ ]:
import json
import logging
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from bs4 import BeautifulSoup
from pymongo import MongoClient, UpdateOne
import ollama
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)       
logging.getLogger("urllib3").setLevel(logging.WARNING)     
logging.getLogger("pymongo").setLevel(logging.WARNING)     

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "sistema_medico_procesado"
COLECCION_NAME = "reportes_raw1"
CAMPO_HTML_ORIGEN = "resto_html"

# Configuración optimizada para procesamiento concurrente con Ollama en paralelo
NUM_HILOS = 4              
TAMAÑO_BLOQUE_MEMORIA = 500 

def limpiar_html_clinico(html_content):
    """Elimina etiquetas HTML (tablas, figuras) y limpia espacios en blanco para optimizar tokens."""
    if not html_content or str(html_content) == 'nan': 
        return ""
    
    # Reemplazo de espacios HTML comunes
    html_content = html_content.replace("&nbsp;", " ").replace("&NBSP;", " ")
    
    # Parseo estricto del HTML
    soup = BeautifulSoup(html_content, "lxml")
    for script in soup(["script", "style"]): 
        script.extract()
    
    # Extracción de texto plano limpio
    texto = soup.get_text(separator=" ")
    texto_limpio = re.sub(r'\s+', ' ', texto).strip()
    return texto_limpio

def procesar_con_ollama(texto_medico):
    """Envía el texto limpio a Ollama y extrae el JSON usando expresiones regulares de forma segura."""
    if not texto_medico: 
        return {"error": "Reporte vacío"}
    
    prompt_sistema = """Eres un extractor de datos médicos profesional. Convierte el texto clínico en el siguiente esquema JSON estricto.

ESQUEMA JSON:
{
  "estudios_detallados": [
    {
      "estudio": "Nombre del estudio (ej. COLUMNA CERVICAL)",
      "hallazgos": ["Lista de observaciones"],
      "conclusiones": ["Lista de diagnósticos"]
    }
  ]
}

REGLAS CRÍTICAS:
1. Crea un bloque independiente por cada estudio detectado.
2. Devuelve EXCLUSIVAMENTE el objeto JSON válido. Sin texto explicativo, sin bloques de código markdown."""

    try:
        response = ollama.chat(
            model='llama3.2:3b',
            messages=[
                {'role': 'system', 'content': prompt_sistema},
                {'role': 'user', 'content': texto_medico}
            ],
            format='json',
            options={
                'temperature': 0.0, 
                'num_predict': 512,  # Incrementado ligeramente por si el JSON clínico es muy extenso  
                'num_ctx': 1024,     # Ajustado para dar más margen a textos médicos largos
                'num_gpu': 99,        
                'low_vram': False,
                'keep_alive': "60m"   
            }
        )
        
        respuesta_raw = response['message']['content']
        
        # --- INTEGRACIÓN DE LA LIMPIEZA Y EXTRACCIÓN ROBUSTA ---
        # Busca el primer '{' y el último '}' para ignorar cualquier texto basura exterior
        match = re.search(r'\{.*\}', respuesta_raw, re.DOTALL)
        if match:
            json_puro = match.group(0)
            return json.loads(json_puro)
        else:
            return {"error_ia": "No se localizó la estructura de llaves del JSON en la respuesta", "raw": respuesta_raw}
            
    except json.JSONDecodeError as je:
        return {"error_ia": f"JSON inválido devuelto por el modelo: {str(je)}", "raw": respuesta_raw}
    except Exception as e:
        return {"error_ia": f"Fallo de conexión o inferencia: {str(e)}"}

def validar_estructura_json(json_data):
    """Valida la integridad de las llaves estructurales necesarias para el reporte."""
    if not json_data or "error" in json_data or "error_ia" in json_data: 
        return False
    if "estudios_detallados" not in json_data: 
        return False
    
    if isinstance(json_data["estudios_detallados"], list):
        for item in json_data["estudios_detallados"]:
            if not isinstance(item, dict): 
                return False
            if "estudio" not in item: 
                item["estudio"] = "No especificado"
            
            for seccion in ["hallazgos", "conclusiones"]:
                lista_orig = item.get(seccion, [])
                lista_limpia = []
                if isinstance(lista_orig, list):
                    for x in lista_orig:
                        if isinstance(x, dict):
                            lista_limpia.append(" ".join([str(v) for v in x.values() if v]))
                        elif isinstance(x, str):
                            lista_limpia.append(x)
                item[seccion] = lista_limpia
    else:
        return False
        
    return True

def procesar_tarea_individual(datos_tupla):
    doc_id, html_sucio = datos_tupla
    
    # 1. Forzar la limpieza radical del HTML
    texto_limpio = limpiar_html_clinico(html_sucio)
    
    # DETECTOR TEMPORAL: Si ves etiquetas en tu consola, la variable viene mal desde MongoDB
    if "<" in texto_limpio or "table" in texto_limpio.lower():
        # Limpieza de emergencia por si el parser falló con estructuras complejas
        texto_limpio = BeautifulSoup(texto_limpio, "lxml").get_text(separator=" ")
        texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()

    if not texto_limpio:
        return UpdateOne(
            {"_id": doc_id},
            {"$set": {"procesado_por_ia": True, "tiene_problema_estructural": True, "motivo_fallo": "HTML vacío u omitido", "reporte_estructurado": None}}
        )
    
    # 2. Enviar estrictamente el texto PLANO a Ollama
    resultado_json = procesar_con_ollama(texto_limpio)
    es_valido = validar_estructura_json(resultado_json)
    
    if not es_valido:
        return UpdateOne(
            {"_id": doc_id},
            {"$set": {"procesado_por_ia": True, "tiene_problema_estructural": True, "motivo_fallo": "Error en estructura jerárquica o JSON corrupto", "reporte_estructurado": resultado_json}}
        )
    
    return UpdateOne(
        {"_id": doc_id},
        {"$set": {"reporte_estructurado": resultado_json, "procesado_por_ia": True, "tiene_problema_estructural": False}}
    )

    
    # 2. Llamada a la IA con el texto plano limpio
    resultado_json = procesar_con_ollama(texto_limpio)
    es_valido = validar_estructura_json(resultado_json)
    
    # 3. Clasificación según la validez de la estructura
    if not es_valido:
        return UpdateOne(
            {"_id": doc_id},
            {"$set": {"procesado_por_ia": True, "tiene_problema_estructural": True, "motivo_fallo": "Error en estructura jerárquica o JSON corrupto", "reporte_estructurado": resultado_json}}
        )
    
    return UpdateOne(
        {"_id": doc_id},
        {"$set": {"reporte_estructurado": resultado_json, "procesado_por_ia": True, "tiene_problema_estructural": False}}
    )

def procesar_en_lotes_seguros():
    client = MongoClient(MONGO_URI)
    coleccion = client[DB_NAME][COLECCION_NAME]
    
    # Asegurar índice en MongoDB
    coleccion.create_index([("procesado_por_ia", 1)])
    
    query = {"procesado_por_ia": {"$exists": False}}
    total_pendientes = coleccion.count_documents(query)
    
    logging.info(f"Total de registros pendientes por procesar: {total_pendientes}")
    if total_pendientes == 0: 
        logging.info("¡Todos los registros ya están procesados!")
        client.close()
        return

    print(f"\n🚀 Iniciando extracción masiva optimizada por bloques en RAM...")
    
    with tqdm(total=total_pendientes, desc="Progreso General", unit="doc") as pbar:
        while True:
            documentos_bloque = list(coleccion.find(query, {"_id": 1, CAMPO_HTML_ORIGEN: 1}).limit(TAMAÑO_BLOQUE_MEMORIA))
            
            if not documentos_bloque:
                break
                
            operaciones_acumuladas = []
            
            with ThreadPoolExecutor(max_workers=NUM_HILOS) as executor:
                futuros = [
                    executor.submit(procesar_tarea_individual, (doc["_id"], doc.get(CAMPO_HTML_ORIGEN, "")))
                    for doc in documentos_bloque
                ]
                
                for futuro in as_completed(futuros):
                    try:
                        update_operation = futuro.result()
                        if update_operation:
                            operaciones_acumuladas.append(update_operation)
                    except Exception as e:
                        logging.error(f"Error en procesamiento individual: {e}")
                    finally:
                        pbar.update(1)
            
            if operaciones_acumuladas:
                coleccion.bulk_write(operaciones_acumuladas, ordered=False)
                
            del documentos_bloque
            del operaciones_acumuladas

    client.close()
    logging.info("Proceso masivo completado exitosamente.")

if __name__ == "__main__":
    procesar_en_lotes_seguros()


2026-10-02 05:19:34,831 - INFO - Total de registros pendientes por procesar: 393771



🚀 Iniciando extracción masiva optimizada por bloques en RAM...


Progreso General:   4%|▎         | 14491/393771 [7:56:10<243:29:42,  2.31s/doc]

In [4]:
from pymongo import MongoClient
import numpy as np

client = MongoClient("mongodb://localhost:27017/")
coleccion = client["sistema_medico_procesado"]["reportes_raw1"]

# 1. Muestreo de longitudes de caracteres
cursor = coleccion.find({"resto_html": {"$exists": True}}, {"resto_html": 1}).limit(200)
longitudes = [len(str(doc.get("resto_html", ""))) for doc in cursor]

if longitudes:
    p95 = int(np.percentile(longitudes, 95))
    media = int(np.mean(longitudes))
    # Regla aproximada: 1 token ≈ 4 caracteres en español clínico
    tokens_estimados_p95 = int(p95 / 3.5) 
    
    print("📊 DIAGNÓSTICO DE TEXTO:")
    print(f"   - Longitud media: {media} caracteres.")
    print(f"   - Longitud Percentil 95 (Casos largos): {p95} caracteres.")
    print(f"   - Contexto (num_ctx) sugerido para cubrir el 95% de casos: {tokens_estimados_p95 + 512} tokens.")
else:
    print("❌ No se encontraron registros con el campo 'resto_html'")
client.close()


📊 DIAGNÓSTICO DE TEXTO:
   - Longitud media: 2208 caracteres.
   - Longitud Percentil 95 (Casos largos): 3653 caracteres.
   - Contexto (num_ctx) sugerido para cubrir el 95% de casos: 1555 tokens.


In [1]:
# Ejecuta esto en una celda antes de reiniciar tu script principal para limpiar los 75 errores
from pymongo import MongoClient
coleccion = MongoClient("mongodb://localhost:27017/")["sistema_medico_procesado"]["reportes_raw1"]

resultado = coleccion.update_many(
    {"tiene_problema_estructural": True},
    {"$unset": {"procesado_por_ia": "", "tiene_problema_estructural": "", "motivo_fallo": "", "reporte_estructurado": ""}}
)
print(f"🔄 Registros defectuosos reseteados listos para reprocesar: {resultado.modified_count}")


🔄 Registros defectuosos reseteados listos para reprocesar: 184


recetea todos los registros

In [2]:
from pymongo import MongoClient
coleccion = MongoClient("mongodb://localhost:27017/")["sistema_medico_procesado"]["reportes_raw1"]

# Removemos el flag de TODOS los registros para procesar el lote completo con la nueva lógica relacional
resultado = coleccion.update_many(
    {}, 
    {"$unset": {"procesado_por_ia": "", "tiene_problema_estructural": "", "motivo_fallo": "", "reporte_estructurado": ""}}
)
print(f"🔄 Base de datos reseteada por completo ({resultado.modified_count} registros) para aplicar la extracción indexada por estudio.")


🔄 Base de datos reseteada por completo (0 registros) para aplicar la extracción indexada por estudio.
